In [1]:
import pandas as pd

#### 1. **Загрузить и прочитать _.csv_ файл**

In [2]:
df = pd.read_csv('../data/auto.csv', index_col='ID')

In [3]:
# Вглянем на данные
df

,CarNumber,Make_n_model,Refund,Fines,History
ID,,,,,
0,Y163O8161RUS,Ford Focus,2.0,3200.0,NaN
1,E432XX77RUS,Toyota Camry,1.0,6500.0,NaN
2,7184TT36RUS,Ford Focus,1.0,2100.0,NaN
3,X582HE161RUS,Ford Focus,2.0,2000.0,NaN
4,E34877152RUS,Ford Focus,2.0,6100.0,NaN
...,...,...,...,...,...
926,Y163O8161RUS,Ford Focus,2.0,1600.0,NaN
927,M0309X197RUS,Ford Focus,1.0,22300.0,NaN
928,O673E8197RUS,Ford Focus,2.0,600.0,NaN


#### 2. **Подсчёт методом _count()_**

In [4]:
total_observations = df['CarNumber'].count()
print(f"Количество записей: {total_observations}")

Количество записей: 931


#### 3. **Обработка дубликатов**

In [5]:
df_no_double = df.drop_duplicates(subset=['CarNumber', 'Make_n_model', 'Fines'], keep='last')

In [6]:
# Подсчёт после удаления дубликатов
df_no_double.count()

CarNumber       725
Make_n_model    725
Refund          713
Fines           665
History          65
dtype: int64

#### 4. **Пропущенные значения**

- Количество отсутствующих значений по столбцам

In [7]:
df_no_double.isna().sum()

CarNumber         0
Make_n_model      0
Refund           12
Fines            60
History         660
dtype: int64

- Удаляем неполный столбец

In [8]:
# Удаляем столбец с пропущенными >= 500 с помощью аргумента "thresh"
df_cleaned = df_no_double.dropna(axis=1, thresh=len(df_no_double) - 500)

In [9]:
# Проверим количество пропущенных значений
print(df_cleaned.isna().sum())

CarNumber        0
Make_n_model     0
Refund          12
Fines           60
dtype: int64


- Заполняем столбец *Refund*

In [10]:
df_cleaned.loc[:, 'Refund'] = df_cleaned['Refund'].ffill()

# Проверим количество пропущенных значений по столбцам
df_cleaned.isna().sum()

CarNumber        0
Make_n_model     0
Refund           0
Fines           60
dtype: int64

- Заполняем столбец _Fines_

In [11]:
mean_fines = df_cleaned['Fines'].mean(skipna=True)
df_cleaned.loc[:, 'Fines'] = df_cleaned['Fines'].fillna(mean_fines)

print(df_cleaned.isna().sum())

CarNumber       0
Make_n_model    0
Refund          0
Fines           0
dtype: int64


#### 5. Марка и модель раздельно

- Новые столбцы методом *apply*

In [12]:
def split_make_model(text):
    parts = str(text).split(maxsplit=1)
    return pd.Series({
        'Make': parts[0] if len(parts) > 0 else "None",
        'Model': parts[1] if len(parts) > 1 else "None"
    })


df_cleaned[['Make', 'Model']] = df_cleaned['Make_n_model'].apply(split_make_model)

- Удалить старый столбец

In [13]:
df_cleaned.drop('Make_n_model', axis=1, inplace=True)

In [14]:
df_cleaned.isna().sum()

CarNumber    0
Refund       0
Fines        0
Make         0
Model        0
dtype: int64

- Сохранить в файл в формате *json*

In [16]:
df_cleaned.to_json(
    '../data/auto.json', 
    orient='records', 
    indent=2,
    force_ascii=False)

In [17]:
df2 = pd.read_json("../data/auto.json", orient="records")

In [18]:
df2.count()

CarNumber    725
Refund       725
Fines        725
Make         725
Model        725
dtype: int64